#### aliasing_tests.ipynb
The notebook performs a series of tests to assess the likelihood that a given $Y_{21}$ signal could arise from superposed higher-degree signals aliased to lower-degree due to the sparse and binary sampling represented by a given observational array.

In [ ]:
import os
import numpy as np
import pandas as pd
import pygplates as pygp
import matplotlib.pyplot as plt
import cartopy.crs as crs
from joblib import Parallel, delayed
import methods as m
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
### user-specified inputs ###
t = 20                # time-step to be considered (the younger end of a spectrum)
step = 10             # time-step to be used
l_max = 60            # maximum SH degree of the basis
l_cutoff = 15         # cutoff for SH taper
alpha = 5             # overall regularization strength
smoothness = 1        # degree-dependent regularization exponent
mask_radius_deg = 8.0      # distance from observations to apply static mask
l_array, m_array = m.lm_arrays(l_max)

static_polygons = pygp.FeatureCollection(f'plate_model/static_polygons.gpml')      # static continental polygons
rotation_model = f'plate_model/PHAB25_zero-lon-Africa.rot'                         # rotation file
anchor_plate = 998                                                                 # anchor plate ID: 998 = paleomagnetic frame with zero-longitude Africa assumption

figdir = f'figs/aliasing_tests'   # directory to output aliasing test result figs
os.makedirs(figdir, exist_ok=True)

In [ ]:
def get_flux(t, step):
    t0 = pd.read_csv(f'obs_grids/{t}_Ma.csv')                 # read in observation grids
    t1 = pd.read_csv(f'obs_grids/{t+step}_Ma.csv')
    merged = pd.merge(t0, t1, on='name', suffixes=('_t0', '_t1'))       # merge dataframes
    merged = merged.dropna(subset=['state_t0', 'state_t1'])             # drop NaNs
    merged['diff'] =  merged['state_t1'].astype(float) - merged['state_t0'].astype(float)  # -1 = regression / 1 = transgression
    fluxpts = merged[merged['diff'] != 0]                               # isolate points where there is flux
    lats = fluxpts['lat_t0'].to_numpy()
    lons = fluxpts['lon_t0'].to_numpy()
    flux = fluxpts['diff'].to_numpy()
    return fluxpts, lats, lons, flux

def sh_exp(lats, lons, flux):
    A, l_array = m.sh_design_matrix(lats, lons, l_max)                  # make sh design matrix from observation points
    coeffs_raw = m.sh_fit(A, flux, alpha, l_array, smoothness)          # fit sh coefficients
    rms = np.sqrt(np.mean((A @ coeffs_raw - flux) ** 2))                # compute misfit
    coeffs = m.apply_taper(coeffs_raw, l_array, l_max, l_cutoff)        # filtering
    
    rlats, rlons, field = m.sh_evaluate_global(coeffs, l_array, l_max)              # evaluate smoothed field on global grid
    mask = m.sh_data_mask(rlats, rlons, lats, lons, max_dist_deg=mask_radius_deg)   # mask data-less regions
    degs, pwr = m.sh_power_spectrum(coeffs, l_array, l_max)                         # get power spectrum
    y21_pwr = m.sh_y21_power(coeffs, l_array, m_array)                              # isolate Y21 power
    return A, rlats, rlons, field, mask, degs, pwr, y21_pwr

In [ ]:
fluxpts, lats, lons, flux = get_flux(t, step)
A, rlats, rlons, field, mask, degs, pwr, y21_pwr = sh_exp(lats, lons, flux)

In [ ]:
polygons = m.reconstruct_polygons(static_polygons, rotation_model, t, anchor_plate=anchor_plate)
fig = plt.figure(figsize=(16, 5))
ax1 = fig.add_subplot(1, 2, 1, projection=crs.Robinson())
m.plot_data_and_sh(polygons, rlats, rlons, field=field, observations=fluxpts, mask=mask, ax=ax1)

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(degs, pwr, color='black', linewidth=1.5, label='fitted S($l$)')
ax2.scatter([2], [y21_pwr], facecolor='none', edgecolor='red', s=70, linewidth=2, zorder=5, label=r'$Y₂₁$ only')
ax2.axvline(2,  color='dodgerblue', linestyle='--', linewidth=1, label='$l$=2')
ax2.set_xlabel('SH degree $l$'); ax2.set_ylabel(r'Power S($l$)')
ax2.set_yscale('log')
ax2.legend(fontsize=10)
plt.savefig(f'{figdir}/SH_obs_{t}.pdf')
plt.savefig(f'{figdir}/SH_obs_{t}.png')
plt.close()

In [ ]:
null_l_center = 40
null_l_width = 15
seed = 0

rng = np.random.default_rng(seed)
rng_pwr = m.gen_spec(l_max, null_l_center, null_l_width)   # get random synthetic power spectra
rng_c = m.sample_coeffs(rng_pwr, l_array, m_array, zero_y21=True, rng=rng)  # draw coefficients from spectra 
rng_field = A @ rng_c                          # continuous random field at observation sites
rng_y = np.where(rng_field >= 0, 1.0, -1.0)    # flatten to ±1 by sign of the field

_, rlats, rlons, s_field, _, s_degs, s_pwr, s_y21_pwr = sh_exp(lats, lons, rng_y)

In [ ]:
fig = plt.figure(figsize=(14, 10))
ax1 = fig.add_subplot(2, 2, 1)
c_norm = rng_pwr / rng_pwr.max() if rng_pwr.max() > 0 else rng_pwr
ax1.plot(np.arange(l_max+1), c_norm, marker='o', markersize=5, color='k')
ax1.axvline(2, color='crimson', linestyle='--', linewidth=1)
ax1.axvline(null_l_center, color='k', linestyle='--', linewidth=0.8)
ax1.set_xlabel('SH degree $l$'); ax1.set_ylabel(r'Normalized power S($l$)')

ax2 = fig.add_subplot(2, 2, 2, projection=crs.Robinson())
y_df = pd.DataFrame({'lat_t0': lats, 'lon_t0': lons, 'diff': rng_y.astype(int)})
m.plot_data_and_sh(polygons, rlats, rlons, field=None, observations=y_df, mask=mask, ax=ax2)
ax3 = fig.add_subplot(2, 2, 3, projection=crs.Robinson())
m.plot_data_and_sh(polygons, rlats, rlons, field=s_field, observations=None, mask=mask, ax=ax3)

ax4 = fig.add_subplot(2, 2, 4)
ax4.plot(s_degs, s_pwr, color='black', linewidth=1.5, label='fitted S($l$), synthetic')
ax4.plot(degs, pwr, color='black', linestyle='--', linewidth=1.5, alpha=0.5, label='fitted S($l$), observed')
ax4.scatter([2], [s_y21_pwr], facecolor='none', edgecolor='red', s=70, linewidth=2, label=r'$Y₂₁$ only, synthetic')
ax4.scatter([2], [y21_pwr], facecolor='none', edgecolor='grey', s=70, linewidth=2, alpha=0.5, label=r'$Y₂₁$ only, observed')
ax4.axvline(2,  color='dodgerblue', linestyle='--', linewidth=1, label='$l$=2', zorder=0)
ax4.set_xlabel('SH degree $l$'); ax4.set_ylabel(r'Power S($l$)')
ax4.set_yscale('log')
ax4.legend(fontsize=10)
plt.savefig(f'{figdir}/SH_syn_{t}.pdf')
plt.savefig(f'{figdir}/SH_syn_{t}.png')
plt.close()

In [ ]:
TPW_intervals = [(20, 30), (90, 100), (140, 150), (190, 200)]

def process_interval(t0, t1):
    fluxpts, lats, lons, flux = get_flux(t0, t1-t0)
    A_t, rlats, rlons, field, mask, degs, pwr, y21_pwr = sh_exp(lats, lons, flux)
    
    null = m.null_y21_sh(A_t, l_array, m_array, l_max, flux, n_iter, null_l_center, null_l_width, 
                       alpha, smoothness, l_cutoff, zero_y21=True, seed=t0)
        
    p_value = float(np.mean(null >= y21_pwr))
    return {'t': t0,
            'n': len(null),
            'p_obs': y21_pwr,
            'null_mean': float(null.mean()),
            'null_95': float(np.quantile(null, 0.95)),
            'null_max': float(null.max()),
            'p_value': p_value,
            'null': null}

n_iter = 10000
results = Parallel(n_jobs=100, backend='loky')(delayed(process_interval)(t0, t1) for (t0, t1) in TPW_intervals) # run in parallel
null_powers = {r['t']: r['null'] for r in results}
p_obs_y21  = {r['t']: r['p_obs'] for r in results}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=False)
for ax, (t0, t1) in zip(axes.ravel(), TPW_intervals):
    nulls = null_powers[t0]
    obs   = p_obs_y21[t0]
    p_val = float(np.mean(nulls >= obs))
    q95   = float(np.quantile(nulls, 0.95))

    ax.hist(nulls, bins=60, color='grey', alpha=0.7, edgecolor='none', label='null (no $Y$₂₁ in spectrum)')
    ax.axvline(obs, color='crimson', linewidth=2, label=f'observed P($Y$₂₁) = {obs:.4f}')
    ax.axvline(q95, color='black', linestyle='--', linewidth=1, label=f'null 95th = {q95:.4f}')
    ax.set_title(f'{t0}–{t1} Ma   p = {p_val:.4g}   (N={len(nulls)})', fontsize=11)
    ax.set_xlabel('recovered $Y$₂₁ power'); ax.set_ylabel('count')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f'{figdir}/MC_results_{null_l_center}_{null_l_width}.pdf', bbox_inches='tight')
plt.savefig(f'{figdir}/MC_results_{null_l_center}_{null_l_width}.png', bbox_inches='tight')
plt.show()